# Retriever

In [53]:
from database import DatabaseClient
class RetrieverClient:
    
    def __init__(self, folder_path):
        self.__database = DatabaseClient(folder_path)

    def __retriever(self, text=None, image_path=None):
        if image_path != None and text != None:
            return self.__database.search_with_image(image_path) + self.database.search_with_text(text)
        elif text == None:
            return self.__database.search_with_image(image_path)
        elif image_path == None:
            return self.__database.search_with_text(text)
    
    def __handle_properties(self, properties):
        response =  {
            'text': [],
            'image': []
        }
        for property in properties:
            if property["media_type"] == "text" and property not in response['text']:
                response['text'].append(property)
            if property["media_type"] == "image" and property not in response['image']:
                response.append(property)
        return response
    
    def search(self, text=None, image_path=None):
        return self.__handle_properties(self.__retriever(text, image_path))
    
    def close_database_connection(self):
        self.__database.close_connection()
        

In [54]:
# retriever = RetrieverClient(r"C:\Users\Anush\Desktop\Christ\Specialization Project\Localinsight\text-only-test\documents")
# retireved = retriever.search("Alice")

In [55]:
# retireved['text'][0]

# OfflineChat

In [ ]:
from ollama import chat
import RetrieverClient

class OfflineChat:
    
    def __init__(self, folder_path, model="phi3"):
        self.messages = []
        self.model = model
        self.retriever = RetrieverClient(folder_path)
    
    def __append_user_message(self, user_query, search_result):
        content = ""
        image_paths = []
        for text_properties in search_result['text']:
            content += text_properties['text'] + "\n"
        for image_properties in search_result['image']:
            image_paths.append(image_properties['path'])
        query_with_context = f"""Given Context: {content}
                                 Query: {user_query}"""
        self.messages.append({"role": "user", "content": query_with_context, "images": image_paths})
    
    def append_assistant_message(self, content):
        self.messages.append({"role": "assistant", "content" : content})  
         
    def get_assistant_response(self, user_text=None, user_image_path=None):
        """
        Usage:
            for chunk in get_assistant_response(...):
                assistant_response += chunk["message"]["content"]
                print(chunk["message"]["content"], end="", flush=True)
            offline_chat.append_assistant_message(assistant_response)
        """
        search_result = self.retriever.search(text=user_text, image_path=user_image_path)
        self.__append_user_message(user_text, search_result)
        return chat(self.model, self.messages, stream=True, options={"temperature":0})


# OnlineChat

In [ ]:
import google.generativeai as genai
from IPython.display import Image
import RetrieverClient

class OnlineChat:
    def __init__(self):
        self.__chat = self.__initiate_chat(api_key=)
         
    def __initiate_chat(self, api_key, model_name="gemini-1.5-flash-001"):
        genai.configure(api_key)
        model = genai.GenerativeModel(model_name=model_name)
        return model.start_chat()
    
    def __create_user_message(self, user_text, search_result):
        content = ""
        images = []
        for text_properties in search_result['text']:
            content += text_properties['text'] + "\n"
        for image_properties in search_result['image']:
            images.append(Image(image_properties['path']))
        query_with_context = f"""Given Context: {content}
                                 Query: {user_text}"""
        return [query_with_context, images]
    
    def get_assistant_response(self, user_text=None, user_image_path=None):
        search_result = self.retriever.search(text=user_text, image_path=user_image_path)
        return self.__chat(self.__create_user_message(user_text, search_result))    

# ChatClient